# Generaci?n de PDFs comparativos (best_recall vs best_general)
Reentrena solo las configuraciones seleccionadas usando datasets ya procesados en `data/intermediate/05_seleccion` (sin reprocesar pipeline completo).

In [ ]:
from __future__ import annotations

import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc, precision_recall_curve, average_precision_score
)
from sklearn.preprocessing import label_binarize, StandardScaler

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neural_network import MLPClassifier

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN
from imblearn.pipeline import Pipeline

REPO_ROOT = Path('.').resolve()
INPUT_BASE = REPO_ROOT / 'data' / 'intermediate' / '05_seleccion'
RUN_ID = datetime.today().strftime('%Y%m%d')
OUT_BASE = REPO_ROOT / 'reports' / '15_pdf_comparacion_ganadores' / RUN_ID
OUT_BASE.mkdir(parents=True, exist_ok=True)

TARGET_CLASS = None  # None => clase minima

BALANCE_METHODS = {
    'NONE': None,
    'SMOTE': SMOTE(random_state=42),
    'UNDER': RandomUnderSampler(random_state=42),
    'SMOTEENN': SMOTEENN(random_state=42),
}

SUMMARY_FILES = {
    'logistic': sorted((REPO_ROOT / 'reports' / '06_modelo_logistic').glob('**/resumen_modelos_logistic_regression.csv'))[-1],
    'xgboost': sorted((REPO_ROOT / 'reports' / '08_modelo_xgboost').glob('**/resumen_modelos_xgboost.csv'))[-1],
    'lightgbm': sorted((REPO_ROOT / 'reports' / '09_modelo_lightgbm').glob('**/resumen_modelos_lightgbm.csv'))[-1],
    'mlp': sorted((REPO_ROOT / 'reports' / '10_modelo_redmlp').glob('**/resumen_modelos_redmlp.csv'))[-1],
}

input_candidates = sorted([p for p in INPUT_BASE.iterdir() if p.is_dir()])
if not input_candidates:
    raise FileNotFoundError(f'No hay subdirectorios en {INPUT_BASE}')
INPUT_PATH = input_candidates[-1]
print('INPUT_PATH:', INPUT_PATH)
print('OUT_BASE:', OUT_BASE)
print('SUMMARY_FILES:', {k: str(v) for k,v in SUMMARY_FILES.items()})


In [ ]:
def resolve_target_class(y, target):
    classes = list(pd.Series(y).unique())
    if target is None:
        return int(pd.Series(y).min())
    if target in classes:
        return int(target)
    if str(target) in [str(c) for c in classes]:
        for c in classes:
            if str(c) == str(target):
                return int(c)
    return int(classes[0])

def build_logistic_pipeline(balance_name):
    balancer = BALANCE_METHODS[balance_name]
    class_weight = None if balancer is not None else 'balanced'
    steps = []
    if balancer is not None:
        steps.append(('balance', balancer))
    steps.append(('model', LogisticRegression(max_iter=1000, solver='lbfgs', class_weight=class_weight)))
    return Pipeline(steps), False

def build_xgboost_pipeline(balance_name):
    balancer = BALANCE_METHODS[balance_name]
    steps = []
    if balancer is not None:
        steps.append(('balance', balancer))
    steps.append(('model', XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        use_label_encoder=False, eval_metric='mlogloss', random_state=42, n_jobs=-1
    )))
    return Pipeline(steps), True

def build_lightgbm_pipeline(balance_name, num_classes):
    balancer = BALANCE_METHODS[balance_name]
    params = dict(
        boosting_type='gbdt',
        objective='multiclass' if num_classes > 2 else 'binary',
        n_estimators=180, learning_rate=0.05,
        num_leaves=31, max_depth=-1,
        subsample=0.85, colsample_bytree=0.9,
        max_bin=63, min_child_samples=20,
        reg_alpha=0.1, reg_lambda=1.0,
        bagging_freq=1, n_jobs=-1, random_state=42,
    )
    if num_classes > 2:
        params['num_class'] = num_classes
    steps = []
    if balancer is not None:
        steps.append(('balance', balancer))
    steps.append(('model', LGBMClassifier(**params)))
    return Pipeline(steps), True

def build_mlp_pipeline(balance_name, num_classes):
    balancer = BALANCE_METHODS[balance_name]
    hidden = (96, 48) if num_classes > 2 else (64, 32)
    steps = []
    if balancer is not None:
        steps.append(('balance', balancer))
    steps.append(('scale', StandardScaler(with_mean=False)))
    steps.append(('model', MLPClassifier(
        hidden_layer_sizes=hidden, activation='relu', solver='adam',
        learning_rate_init=0.002, max_iter=60, batch_size=128, alpha=1e-4,
        early_stopping=True, n_iter_no_change=6, validation_fraction=0.12,
        shuffle=True, random_state=42, verbose=False
    )))
    return Pipeline(steps), True

def pick_configs(summary_df):
    best_recall = summary_df.sort_values(['cv_recall_target','cv_macro_f1'], ascending=False).iloc[0]
    # 'general' definido como mejor balance global en CV
    best_general = summary_df.sort_values(['cv_macro_f1','cv_recall_target'], ascending=False).iloc[0]
    return best_recall, best_general

def generate_pdf(y_true, y_pred, y_proba, class_order, out_pdf):
    cm = confusion_matrix(y_true, y_pred, labels=class_order)
    y_bin = label_binarize(y_true, classes=class_order)

    with PdfPages(out_pdf) as pdf:
        # Confusion matrix
        ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_order).plot(cmap='Blues')
        plt.title('Matriz de confusi?n')
        plt.tight_layout()
        pdf.savefig(); plt.close()

        # ROC
        plt.figure(figsize=(8,6))
        for i, cls in enumerate(class_order):
            if y_bin.shape[1] == 1:
                # binary edge case
                if len(class_order) == 2:
                    yy = (y_true == cls).astype(int)
                    pp = y_proba[:, i] if y_proba.shape[1] > i else y_proba[:, 0]
                else:
                    continue
            else:
                yy = y_bin[:, i]
                pp = y_proba[:, i]
            if len(np.unique(yy)) < 2:
                continue
            fpr, tpr, _ = roc_curve(yy, pp)
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, label=f'Clase {cls} (AUC={roc_auc:.3f})')
        plt.plot([0,1],[0,1],'k--')
        plt.xlabel('FPR')
        plt.ylabel('TPR')
        plt.title('Curvas ROC por clase')
        plt.legend(loc='best')
        plt.tight_layout()
        pdf.savefig(); plt.close()

        # PR
        plt.figure(figsize=(8,6))
        for i, cls in enumerate(class_order):
            if y_bin.shape[1] == 1:
                if len(class_order) == 2:
                    yy = (y_true == cls).astype(int)
                    pp = y_proba[:, i] if y_proba.shape[1] > i else y_proba[:, 0]
                else:
                    continue
            else:
                yy = y_bin[:, i]
                pp = y_proba[:, i]
            if len(np.unique(yy)) < 2:
                continue
            precision, recall, _ = precision_recall_curve(yy, pp)
            pr_auc = average_precision_score(yy, pp)
            plt.plot(recall, precision, label=f'Clase {cls} (AP={pr_auc:.3f})')
        plt.xlabel('Recall')
        plt.ylabel('Precision')
        plt.title('Curvas Precision-Recall por clase')
        plt.legend(loc='best')
        plt.tight_layout()
        pdf.savefig(); plt.close()

def run_one(model_key, base_name, balance_name, tag):
    X_train = pd.read_parquet(INPUT_PATH / f'X_train_{base_name}.parquet')
    X_test = pd.read_parquet(INPUT_PATH / f'X_test_{base_name}.parquet')
    y_train = pd.read_parquet(INPUT_PATH / f'y_train_{base_name}.parquet').squeeze()
    y_test = pd.read_parquet(INPUT_PATH / f'y_test_{base_name}.parquet').squeeze()

    target_class = resolve_target_class(y_train, TARGET_CLASS)

    if model_key == 'logistic':
        model, needs_adj = build_logistic_pipeline(balance_name)
    elif model_key == 'xgboost':
        model, needs_adj = build_xgboost_pipeline(balance_name)
    elif model_key == 'lightgbm':
        y_min = int(y_train.min())
        num_classes = int(pd.Series(y_train - y_min).nunique())
        model, needs_adj = build_lightgbm_pipeline(balance_name, num_classes)
    elif model_key == 'mlp':
        y_min = int(y_train.min())
        num_classes = int(pd.Series(y_train - y_min).nunique())
        model, needs_adj = build_mlp_pipeline(balance_name, num_classes)
    else:
        raise ValueError(model_key)

    if needs_adj:
        y_min = int(y_train.min())
        y_train_fit = y_train - y_min
        y_test_fit = y_test - y_min
        model.fit(X_train, y_train_fit)
        y_pred = model.predict(X_test) + y_min
        y_proba = model.predict_proba(X_test)
        class_order = np.array(sorted(pd.Series(y_train).unique()))
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)
        class_order = np.array(sorted(pd.Series(y_train).unique()))

    out_dir = OUT_BASE / model_key
    out_dir.mkdir(parents=True, exist_ok=True)
    out_pdf = out_dir / f'pdf_{tag}_{base_name}_{balance_name}.pdf'
    generate_pdf(y_test, y_pred, y_proba, class_order, out_pdf)

    rep = pd.DataFrame(classification_report(y_test, y_pred, output_dict=True, zero_division=0)).T
    rep.to_csv(out_dir / f'classification_report_{tag}_{base_name}_{balance_name}.csv')

    return {
        'modelo': model_key,
        'tag': tag,
        'base_name': base_name,
        'balanceo': balance_name,
        'target_class': target_class,
        'macro_f1_test': float(rep.loc['macro avg','f1-score']),
        'weighted_f1_test': float(rep.loc['weighted avg','f1-score']),
        'accuracy_test': float(rep.loc['accuracy','precision']) if 'accuracy' in rep.index else float('nan'),
        'pdf_path': str(out_pdf),
    }


In [ ]:
results = []
selection_rows = []

for model_key, summary_file in SUMMARY_FILES.items():
    df = pd.read_csv(summary_file)
    best_recall, best_general = pick_configs(df)

    br_base, br_bal = str(best_recall['modelo']), str(best_recall['balanceo'])
    bg_base, bg_bal = str(best_general['modelo']), str(best_general['balanceo'])

    selection_rows.append({
        'modelo': model_key,
        'summary_file': str(summary_file),
        'best_recall_base': br_base,
        'best_recall_balanceo': br_bal,
        'best_recall_cv_recall_target': float(best_recall['cv_recall_target']),
        'best_recall_cv_macro_f1': float(best_recall['cv_macro_f1']),
        'best_general_base': bg_base,
        'best_general_balanceo': bg_bal,
        'best_general_cv_recall_target': float(best_general['cv_recall_target']),
        'best_general_cv_macro_f1': float(best_general['cv_macro_f1']),
    })

    # Generar PDF best_recall
    results.append(run_one(model_key, br_base, br_bal, 'best_recall'))

    # Generar PDF best_general (si difiere)
    if (bg_base, bg_bal) != (br_base, br_bal):
        results.append(run_one(model_key, bg_base, bg_bal, 'best_general'))
    else:
        results.append({
            'modelo': model_key,
            'tag': 'best_general',
            'base_name': bg_base,
            'balanceo': bg_bal,
            'note': 'Coincide con best_recall; se reutiliza el mismo PDF',
        })

sel_df = pd.DataFrame(selection_rows)
res_df = pd.DataFrame(results)

sel_df.to_csv(OUT_BASE / 'seleccion_ganadores.csv', index=False)
res_df.to_csv(OUT_BASE / 'resultados_pdf_generados.csv', index=False)

print('Guardado en:', OUT_BASE)
display(sel_df)
display(res_df)
